# CNN Experiment and Evaluation

## Setup & Import

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import tensorflow as tf
import ast
from tensorflow import keras
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

# Import custom layers
from cnn.layers.conv2d import Conv2D
from cnn.layers.locallyconnected2d import LocallyConnected2D
from cnn.layers.pooling2d import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.layers.flatten import Flatten
from cnn.layers.dense import Dense
from cnn.utility import batch_loader

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


## Load Training Results

In [ ]:
# Load hasil pelatihan
results_df = pd.read_csv('../results/cnn/experiment_cnn.csv')

results_df['filters'] = results_df['filters'].apply(ast.literal_eval)
results_df['kernel_size'] = results_df['kernel_size'].apply(ast.literal_eval)

results_sorted = results_df.sort_values('f1_macro', ascending=False)

print("Top 5 Models by Macro F1-Score:")
print("="*80)
print(results_sorted[[
    'experiment_id', 'model_name', 'f1_macro', 'test_accuracy',
    'num_conv_layers', 'filters', 'kernel_size', 'pooling_type', 'num_params'
]].head())

# Best model
best_model_row = results_sorted.iloc[0]
best_model_name = best_model_row['model_name']
best_model_path = f'../models/cnn/{best_model_name}.keras'

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*80}")
print(f"F1 Score: {best_model_row['f1_macro']:.4f}")
print(f"Accuracy: {best_model_row['test_accuracy']:.4f}")
print(f"Layers: {best_model_row['num_conv_layers']}")
print(f"Filters: {best_model_row['filters']}")
print(f"Kernel: {best_model_row['kernel_size']}")
print(f"Pooling: {best_model_row['pooling_type']}")
print(f"Parameters: {best_model_row['num_params']:,}")
print(f"{'='*80}")

Top 5 Models by Macro F1-Score:
    experiment_id model_name  f1_macro  test_accuracy  num_conv_layers  \
14             15   model_15  0.819792       0.819000                3   
10             11   model_11  0.798122       0.797333                3   
15             16   model_16  0.792097       0.790667                3   
11             12   model_12  0.763133       0.763333                3   
12             13   model_13  0.758414       0.755667                3   

      filters kernel_size pooling_type  num_params  
14  [64, 128]      [5, 5]          max      636806  
10   [32, 64]      [5, 5]          max      165254  
15  [64, 128]      [5, 5]      average      636806  
11   [32, 64]      [5, 5]      average      165254  
12  [64, 128]      [3, 3]          max      240518  

BEST MODEL: model_15
F1 Score: 0.8198
Accuracy: 0.8190
Layers: 3
Filters: [64, 128]
Kernel: [5, 5]
Pooling: max
Parameters: 636,806


## Load Dataset

In [3]:
# Dataset configuration
DATASET_PATH = Path('../dataset')
IMG_SIZE = (150, 150)
CHANNELS = 3
RANDOM_SEED = 42

def load_images_from_directory(directory, img_size=(128, 128), channels=3):
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")
    
    class_folders = sorted([d for d in directory.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_folders]
    
    all_image_paths = []
    all_labels = []
    
    for class_idx, class_folder in enumerate(class_folders):
        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            image_paths.extend(list(class_folder.glob(ext)))
            image_paths.extend(list(class_folder.glob(ext.upper())))
        
        image_paths = list({Path(p).resolve() for p in image_paths})
        image_paths = sorted([str(p) for p in image_paths])
        
        all_image_paths.extend(image_paths)
        all_labels.extend([class_idx] * len(image_paths))
    
    images = batch_loader(all_image_paths, h=img_size[0], w=img_size[1], c=channels)
    labels = np.array(all_labels)
    
    return images, labels, class_names

# Load test set
test_dir = DATASET_PATH / 'seg_test' / 'seg_test'
print("Loading test set...")
X_test, y_test, class_names = load_images_from_directory(test_dir, IMG_SIZE, CHANNELS)
print(f"Test set shape: {X_test.shape}")
print(f"Classes: {class_names}")

Loading test set...
Test set shape: (3000, 150, 150, 3)
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


## Load Best Keras Model

In [4]:
# Load best model
keras_model = keras.models.load_model(best_model_path)
print(f"Loaded model: {best_model_path}")
print("\nModel Summary:")
keras_model.summary()

# Evaluate Keras model
print("\nEvaluating Keras model on test set...")
keras_loss, keras_acc = keras_model.evaluate(X_test, y_test, verbose=0)
keras_pred = keras_model.predict(X_test, verbose=0)
keras_pred_classes = np.argmax(keras_pred, axis=1)
keras_f1 = f1_score(y_test, keras_pred_classes, average='macro')

print(f"\nKeras Model Performance:")
print(f"  Accuracy: {keras_acc:.4f}")
print(f"  Loss: {keras_loss:.4f}")
print(f"  Macro F1: {keras_f1:.4f}")

Loaded model: ../models/cnn/model_15.keras

Model Summary:


Model: "cnn_3layers"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv2D)                 │ (None, 150, 150, 64)   │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, 75, 75, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_3 (Conv2D)                 │ (None, 37, 37, 128)    │       409,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_3 (MaxPooling2D)        │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_pool                     │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,910,420 (7.29 MB)

 Trainable params: 636,806 (2.43 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,273,614 (4.86 MB)


Evaluating Keras model on test set...

Keras Model Performance:
  Accuracy: 0.8190
  Loss: 0.4996
  Macro F1: 0.8198


In [5]:
import os
import multiprocessing

n_cores = str(multiprocessing.cpu_count())
os.environ["OPENBLAS_NUM_THREADS"] = n_cores  # ← yang relevan untuk numpy kamu
os.environ["OMP_NUM_THREADS"]      = n_cores

import numpy as np

In [6]:
from threadpoolctl import threadpool_info
for pool in threadpool_info():
    print(f"{pool['user_api']}: {pool['num_threads']} threads — {pool['filepath']}")

blas: 16 threads — C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy.libs\libscipy_openblas64_-860d95b1c38e637ce4509f5fa24fbf2a.dll
openmp: 16 threads — C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\.libs\vcomp140.dll
blas: 16 threads — C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy.libs\libscipy_openblas-64eda39e79589aedb16f58e5547eb599.dll


## Build Custom Model (From Scratch - Shared Parameters)

In [12]:
class CustomCNN:
    """CNN from scratch using custom layers"""
    
    def __init__(self, keras_model):
        self.layers = []
        self.layer_names = []
    
        for layer in keras_model.layers:
            layer_name = layer.name
            layer_type = type(layer).__name__
            
            print(f"Loading layer: {layer_name} ({layer_type})")
            
            if 'conv' in layer_name:
                custom_layer = Conv2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'maxpool' in layer_name:
                custom_layer = MaxPooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'avgpool' in layer_name:
                custom_layer = AveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'global' in layer_name:
                custom_layer = GlobalAveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dense' in layer_name or 'output' in layer_name:
                custom_layer = Dense(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dropout' in layer_name:
                print(f"  Skipping dropout layer ")
                continue
            
            else:
                print(f"  Unknown layer {layer_type}")
        
        print(f"\nTotal custom layers loaded: {len(self.layers)}")
    
    def forward(self, x, verbose=False):
        """Forward propagation"""
        for i, (layer, name) in enumerate(zip(self.layers, self.layer_names)):
            x = layer.forward(x)
            if verbose:
                print(f"  Layer {i+1} ({name}): output shape = {x.shape}")
        return x
    
    def predict(self, X, batch_size=32, verbose=True):
        n_samples = X.shape[0]
        n_batches = (n_samples - 1) // batch_size + 1
        predictions = []

        for i in range(0, n_samples, batch_size):
            batch = X[i:i+batch_size]
            batch_idx = i // batch_size + 1

            if verbose:
                print(f"\nBatch {batch_idx}/{n_batches}")

            predictions.append(self.forward(batch, verbose=(batch_idx == 1)))

        return np.vstack(predictions)

# Build custom model
print("Building custom CNN from scratch...")
print("="*80)
custom_model_shared = CustomCNN(keras_model)
print("="*80)

Building custom CNN from scratch...
Loading layer: conv_1 (Conv2D)
Loading layer: maxpool_1 (MaxPooling2D)
Loading layer: conv_2 (Conv2D)
Loading layer: maxpool_2 (MaxPooling2D)
Loading layer: conv_3 (Conv2D)
Loading layer: maxpool_3 (MaxPooling2D)
Loading layer: global_pool (GlobalAveragePooling2D)
Loading layer: dense_1 (Dense)
Loading layer: dropout (Dropout)
  Skipping dropout layer 
Loading layer: output (Dense)

Total custom layers loaded: 9


## Test Custom Model (Shared Parameters)

In [13]:
print("\nTesting...")

custom_pred_full = custom_model_shared.predict(X_test, batch_size=8, verbose=True)
custom_classes_full = np.argmax(custom_pred_full, axis=1)
custom_f1 = f1_score(y_test, custom_classes_full, average='macro')
custom_acc = np.mean(custom_classes_full == y_test)

print("\nCustom Model (Shared) Performance:")
print(f"  Accuracy: {custom_acc:.4f}")
print(f"  Macro F1: {custom_f1:.4f}")

print("\n" + "="*80)
print("KERAS vs CUSTOM (Shared) COMPARISON")
print("="*80)
print(f"Keras    - Accuracy: {keras_acc:.4f}, F1: {keras_f1:.4f}")
print(f"Custom   - Accuracy: {custom_acc:.4f}, F1: {custom_f1:.4f}")
print(f"Difference - Accuracy: {abs(keras_acc - custom_acc):.6f}, F1: {abs(keras_f1 - custom_f1):.6f}")
print("="*80)


Testing...

Batch 1/375
  Layer 1 (conv_1): output shape = (8, 150, 150, 64)
  Layer 2 (maxpool_1): output shape = (8, 75, 75, 64)
  Layer 3 (conv_2): output shape = (8, 75, 75, 128)
  Layer 4 (maxpool_2): output shape = (8, 37, 37, 128)
  Layer 5 (conv_3): output shape = (8, 37, 37, 128)
  Layer 6 (maxpool_3): output shape = (8, 18, 18, 128)
  Layer 7 (global_pool): output shape = (8, 128)
  Layer 8 (dense_1): output shape = (8, 128)
  Layer 9 (output): output shape = (8, 6)

Batch 2/375

Batch 3/375

Batch 4/375

Batch 5/375

Batch 6/375

Batch 7/375

Batch 8/375

Batch 9/375

Batch 10/375

Batch 11/375

Batch 12/375

Batch 13/375

Batch 14/375

Batch 15/375

Batch 16/375

Batch 17/375

Batch 18/375

Batch 19/375

Batch 20/375

Batch 21/375

Batch 22/375

Batch 23/375

Batch 24/375

Batch 25/375

Batch 26/375

Batch 27/375

Batch 28/375

Batch 29/375

Batch 30/375

Batch 31/375

Batch 32/375

Batch 33/375

Batch 34/375

Batch 35/375

Batch 36/375

Batch 37/375

Batch 38/375

Batch 3

## Classification Report (Keras vs Custom Shared)

In [13]:
print("\n" + "="*80)
print("CLASSIFICATION REPORT - KERAS (SHARED)")
print("="*80)
print(classification_report(y_test, keras_pred_classes, target_names=class_names))

print("\n" + "="*80)
print("CLASSIFICATION REPORT - CUSTOM (SHARED)")
print("="*80)
print(classification_report(y_test, custom_classes_full, target_names=class_names))


CLASSIFICATION REPORT - KERAS (SHARED)
              precision    recall  f1-score   support

   buildings       0.70      0.88      0.78       437
      forest       0.96      0.95      0.96       474
     glacier       0.83      0.73      0.78       553
    mountain       0.78      0.80      0.79       525
         sea       0.78      0.88      0.83       510
      street       0.91      0.69      0.78       501

    accuracy                           0.82      3000
   macro avg       0.83      0.82      0.82      3000
weighted avg       0.83      0.82      0.82      3000


CLASSIFICATION REPORT - CUSTOM (SHARED)
              precision    recall  f1-score   support

   buildings       0.70      0.88      0.78       437
      forest       0.96      0.95      0.96       474
     glacier       0.83      0.73      0.78       553
    mountain       0.78      0.80      0.79       525
         sea       0.78      0.88      0.83       510
      street       0.91      0.69      0.78       5

## Build Custom Model (From Scratch - Non Shared Parameters)

In [7]:
class CustomNonCNN:
    """CNN (Non Shared) from scratch using custom layers"""
    
    def __init__(self, keras_model):
        self.layers = []
        self.layer_names = []
    
        for layer in keras_model.layers:
            layer_name = layer.name
            layer_type = type(layer).__name__
            
            print(f"Loading layer: {layer_name} ({layer_type})")
            
            if 'conv' in layer_name:
                weights = layer.get_weights()
                kernel = weights[0]

                if kernel.ndim == 4:
                    kH, kW, C_in, C_out = kernel.shape
                    kernel_flat = kernel.reshape(-1, C_out)

                    layer._lc_kernel = kernel_flat 
                    layer._lc_bias   = weights[1]

                elif kernel.ndim == 3:
                    layer._lc_kernel = weights[0]
                    layer._lc_bias   = weights[1]

                custom_layer = LocallyConnected2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'maxpool' in layer_name:
                custom_layer = MaxPooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'avgpool' in layer_name:
                custom_layer = AveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'global' in layer_name:
                custom_layer = GlobalAveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dense' in layer_name or 'output' in layer_name:
                custom_layer = Dense(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dropout' in layer_name:
                print(f"  Skipping dropout layer ")
                continue
            
            else:
                print(f"  Unknown layer {layer_type}")
        
        print(f"\nTotal custom layers loaded: {len(self.layers)}")
    
    def forward(self, x, verbose=False):
        """Forward propagation"""
        for i, (layer, name) in enumerate(zip(self.layers, self.layer_names)):
            x = layer.forward(x)
            if verbose:
                print(f"  Layer {i+1} ({name}): output shape = {x.shape}")
        return x
    
    def predict(self, X, batch_size=32, verbose=True):
        n_samples = X.shape[0]
        n_batches = (n_samples - 1) // batch_size + 1
        predictions = []

        for i in range(0, n_samples, batch_size):
            batch = X[i:i+batch_size]
            batch_idx = i // batch_size + 1

            if verbose:
                print(f"\nBatch {batch_idx}/{n_batches}")

            predictions.append(self.forward(batch, verbose=(batch_idx == 1)))

        return np.vstack(predictions)

# Build custom model
print("Building custom Non Shared CNN from scratch...")
print("="*80)
custom_model_nonshared = CustomNonCNN(keras_model)
print("="*80)

Building custom Non Shared CNN from scratch...
Loading layer: conv_1 (Conv2D)
Loading layer: maxpool_1 (MaxPooling2D)
Loading layer: conv_2 (Conv2D)
Loading layer: maxpool_2 (MaxPooling2D)
Loading layer: conv_3 (Conv2D)
Loading layer: maxpool_3 (MaxPooling2D)
Loading layer: global_pool (GlobalAveragePooling2D)
Loading layer: dense_1 (Dense)
Loading layer: dropout (Dropout)
  Skipping dropout layer 
Loading layer: output (Dense)

Total custom layers loaded: 9


## Classification Report (Custom Shared vs Custom Non Shared)

In [9]:
print("\nTesting Non Shared...")

custom_non_pred_full = custom_model_nonshared.predict(X_test, batch_size=8, verbose=True)
custom_non_classes_full = np.argmax(custom_non_pred_full, axis=1)
custom_non_f1 = f1_score(y_test, custom_non_classes_full, average='macro')
custom_non_acc = np.mean(custom_non_classes_full == y_test)

print("\nCustom Model (Shared) Performance:")
print(f"  Accuracy: {custom_non_acc:.4f}")
print(f"  Macro F1: {custom_non_f1:.4f}")


Testing Non Shared...

Batch 1/375
  Layer 1 (conv_1): output shape = (8, 150, 150, 64)
  Layer 2 (maxpool_1): output shape = (8, 75, 75, 64)
  Layer 3 (conv_2): output shape = (8, 75, 75, 128)
  Layer 4 (maxpool_2): output shape = (8, 37, 37, 128)
  Layer 5 (conv_3): output shape = (8, 37, 37, 128)
  Layer 6 (maxpool_3): output shape = (8, 18, 18, 128)
  Layer 7 (global_pool): output shape = (8, 128)
  Layer 8 (dense_1): output shape = (8, 128)
  Layer 9 (output): output shape = (8, 6)

Batch 2/375

Batch 3/375

Batch 4/375

Batch 5/375

Batch 6/375

Batch 7/375

Batch 8/375

Batch 9/375

Batch 10/375

Batch 11/375

Batch 12/375

Batch 13/375

Batch 14/375

Batch 15/375

Batch 16/375

Batch 17/375

Batch 18/375

Batch 19/375

Batch 20/375

Batch 21/375

Batch 22/375

Batch 23/375

Batch 24/375

Batch 25/375

Batch 26/375

Batch 27/375

Batch 28/375

Batch 29/375

Batch 30/375

Batch 31/375

Batch 32/375

Batch 33/375

Batch 34/375

Batch 35/375

Batch 36/375

Batch 37/375

Batch 38/3

In [ ]:
print("\n" + "="*80)
print("Shared vs Non Shared COMPARISON")
print("="*80)
print(f"Shared   - Accuracy: {custom_acc:.4f}, F1: {custom_f1:.4f}")
print(f"Non Shared   - Accuracy: {custom_non_acc:.4f}, F1: {custom_non_f1:.4f}")
print(f"Difference - Accuracy: {abs(custom_non_acc - custom_acc):.6f}, F1: {abs(custom_non_f1 - custom_f1):.6f}")
print("="*80)

print("\n" + "="*80)
print("CLASSIFICATION REPORT - CUSTOM (SHARED)")
print("="*80)
print(classification_report(y_test, custom_classes_full, target_names=class_names))

print("\n" + "="*80)
print("CLASSIFICATION REPORT - CUSTOM (NON SHARED)")
print("="*80)
print(classification_report(y_test, custom_non_classes_full, target_names=class_names))


Custom Model (Shared) Performance:
  Accuracy: 0.8190
  Macro F1: 0.8198

Shared vs Non Shared COMPARISON
Shared   - Accuracy: 0.8190, F1: 0.8198
Non Shared   - Accuracy: 0.8190, F1: 0.8198
Difference - Accuracy: 0.000000, F1: 0.000000

CLASSIFICATION REPORT - CUSTOM (SHARED)
              precision    recall  f1-score   support

   buildings       0.70      0.88      0.78       437
      forest       0.96      0.95      0.96       474
     glacier       0.83      0.73      0.78       553
    mountain       0.78      0.80      0.79       525
         sea       0.78      0.88      0.83       510
      street       0.91      0.69      0.78       501

    accuracy                           0.82      3000
   macro avg       0.83      0.82      0.82      3000
weighted avg       0.83      0.82      0.82      3000


CLASSIFICATION REPORT - CUSTOM (NON SHARED)
              precision    recall  f1-score   support

   buildings       0.70      0.88      0.78       437
      forest       0.96  